In [4]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="In one sentence, what is machine learning?"
)
print(response.text)
!pip install -q -U google-genai

from google import genai
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

print("Gemini API configured successfully!")

Machine learning is a branch of artificial intelligence that enables computers to learn from data, recognize patterns, and make decisions or predictions without being explicitly programmed.
Gemini API configured successfully!


In [9]:
# This is our "document" - replace this with real content later if you want
document_text = """
SafeGuardian is a family safety app built with Flutter and Firebase.
It includes an SOS button that lets a child alert their parent immediately.
When the SOS button is pressed, the app records a brief audio clip to estimate
distress level, then deletes the audio right away - it is never stored.

The app also has a Safety Check feature that scans notification text from
messaging apps like WhatsApp for signs of bullying or threats. Only a risk
category is shared with the parent, never the actual message content.

Study Mode lets parents set a study time window and get alerted if the child
uses distracting apps like YouTube or TikTok heavily during that time.

The app uses Firebase Authentication and Firestore security rules to keep
each family's data completely separate and secure. A parent can also set a
PIN that is required before a child can unlink their own device.
"""

print("Document loaded, length:", len(document_text), "characters")

Document loaded, length: 881 characters


In [10]:
# Split the document into paragraphs (simple chunking by blank lines)
chunks = [chunk.strip() for chunk in document_text.strip().split("\n\n") if chunk.strip()]

print(f"Split into {len(chunks)} chunks:\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()

Split into 4 chunks:

--- Chunk 0 ---
SafeGuardian is a family safety app built with Flutter and Firebase.
It includes an SOS button that lets a child alert their parent immediately.
When the SOS button is pressed, the app records a brief audio clip to estimate
distress level, then deletes the audio right away - it is never stored.

--- Chunk 1 ---
The app also has a Safety Check feature that scans notification text from
messaging apps like WhatsApp for signs of bullying or threats. Only a risk
category is shared with the parent, never the actual message content.

--- Chunk 2 ---
Study Mode lets parents set a study time window and get alerted if the child
uses distracting apps like YouTube or TikTok heavily during that time.

--- Chunk 3 ---
The app uses Firebase Authentication and Firestore security rules to keep
each family's data completely separate and secure. A parent can also set a
PIN that is required before a child can unlink their own device.



In [11]:
# Generate an embedding (a numeric "meaning fingerprint") for each chunk
result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=chunks
)

chunk_embeddings = [embedding.values for embedding in result.embeddings]

print(f"Generated {len(chunk_embeddings)} embeddings")
print(f"Each embedding has {len(chunk_embeddings[0])} numbers")

Generated 4 embeddings
Each embedding has 3072 numbers


In [12]:
import numpy as np

def find_most_relevant_chunk(question, chunks, chunk_embeddings):
    # Embed the question the same way we embedded the chunks
    question_embedding = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[question]
    ).embeddings[0].values

    # Compare the question to each chunk using cosine similarity
    similarities = []
    for chunk_emb in chunk_embeddings:
        similarity = np.dot(question_embedding, chunk_emb) / (
            np.linalg.norm(question_embedding) * np.linalg.norm(chunk_emb)
        )
        similarities.append(similarity)

    # Return the chunk with the highest similarity score
    best_index = np.argmax(similarities)
    return chunks[best_index], similarities[best_index]

# Test it
question = "What happens when the SOS button is pressed?"
best_chunk, score = find_most_relevant_chunk(question, chunks, chunk_embeddings)

print(f"Question: {question}")
print(f"\nMost relevant chunk (similarity: {score:.2f}):")
print(best_chunk)

Question: What happens when the SOS button is pressed?

Most relevant chunk (similarity: 0.61):
SafeGuardian is a family safety app built with Flutter and Firebase.
It includes an SOS button that lets a child alert their parent immediately.
When the SOS button is pressed, the app records a brief audio clip to estimate
distress level, then deletes the audio right away - it is never stored.


In [13]:
def answer_question(question, chunks, chunk_embeddings):
    # Step 1: Retrieve the most relevant chunk
    best_chunk, score = find_most_relevant_chunk(question, chunks, chunk_embeddings)

    # Step 2: Generate an answer using that chunk as context
    prompt = f"""Answer the question using ONLY the information in the context below.
If the answer isn't in the context, say "I don't have that information."

Context:
{best_chunk}

Question: {question}

Answer:"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text

# Test it with a few different questions
questions = [
    "What happens when the SOS button is pressed?",
    "How does Study Mode work?",
    "What color is the app's logo?"  # This isn't in our document - should say it doesn't know
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {answer_question(q, chunks, chunk_embeddings)}")
    print()

Q: What happens when the SOS button is pressed?
A: When the SOS button is pressed, a child alerts their parent immediately, and the app records a brief audio clip to estimate the distress level. The audio clip is then deleted right away and is never stored.

Q: How does Study Mode work?
A: Based on the provided context, Study Mode works by allowing parents to set a study time window and sending them an alert if their child heavily uses distracting apps, such as YouTube or TikTok, during that time.

Q: What color is the app's logo?
A: I don't have that information.



# Document Q&A Bot (RAG)

A retrieval-augmented generation (RAG) system that answers questions about a document using only the information it actually contains — including correctly saying "I don't know" when the answer isn't there.

Built using Google's Gemini API for embeddings and generation.